# Module 7: Patterns in the Crowd

**Case File 03: Will They Show Up?**

You cleaned the evidence in Module 6. Now investigate the patterns before Module 8 attempts prediction. Work from top to bottom. Keep every earlier cell because later cells build on it.


## Before you begin
1. Upload `campus_events_clean.csv` to the Colab Files panel.
2. Run each cell with the play button.
3. Read every printed result and chart label.
4. Use Gemini when prompted, but verify its claim against the table or chart.
5. Never upload real student or school data.


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=196dbe6b-8042-4040-971b-b4b7015315d9"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Import the tools used throughout this notebook.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned file produced by the Module 6 workflow.
df = pd.read_csv('campus_events_clean.csv')
# Make a numeric analysis copy of the yes/no target. It is not an input feature.
df['high_turnout_flag'] = df['high_turnout'].map({'no': 0, 'yes': 1})
print('Rows and columns:', df.shape)
df.head()


## Level 1: Describe a typical event
A mean uses every value. A median is the middle after sorting. Range is maximum minus minimum. The interquartile range, or IQR, describes the middle half of the values and is less sensitive to extremes.


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=9a943215-ec19-4af5-8493-b4b70153153d"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Start with the target-related rate for description only.
# IMPORTANT: attendance_rate and actual_attendance will NOT become model features.
column = 'attendance_rate'
mean_value = df[column].mean()
median_value = df[column].median()
range_value = df[column].max() - df[column].min()
standard_deviation = df[column].std()  # pandas uses the sample SD by default
q1 = df[column].quantile(0.25)
q3 = df[column].quantile(0.75)
iqr = q3 - q1
print(f'Mean: {mean_value:.3f}')
print(f'Median: {median_value:.3f}')
print(f'Range: {range_value:.3f}')
print(f'Sample standard deviation: {standard_deviation:.3f}')
print(f'IQR: {iqr:.3f}')


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=eace37a7-8491-4672-9578-b4b701531705"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# A histogram shows the distribution, not just one summary number.
plt.figure(figsize=(8, 4))
sns.histplot(df['attendance_rate'], bins=12, color='#A32638')
plt.axvline(df['attendance_rate'].median(), color='black', linestyle='--', label='median')
plt.xlabel('Attendance rate')
plt.ylabel('Number of events')
plt.title('How attendance rates are distributed')
plt.legend()
plt.savefig('m7_attendance_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved m7_attendance_distribution.png')


**Evidence note 1:** In a new markdown cell, write: The typical attendance rate was ____. I chose mean or median because ____. The range, sample standard deviation, or IQR shows ____. The distribution also shows ____.


## Level 2: Investigate outliers
The 1.5 × IQR rule flags values for investigation. A flag is not an instruction to delete. An unusual event may be an error, or it may be important evidence.


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=5d486eca-8e68-45be-9252-b4b7015328cd"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Flag possible outliers in a pre-event clue with the IQR rule.
outlier_column = 'previous_similar_attendance'
outlier_q1 = df[outlier_column].quantile(0.25)
outlier_q3 = df[outlier_column].quantile(0.75)
outlier_iqr = outlier_q3 - outlier_q1
lower = outlier_q1 - 1.5 * outlier_iqr
upper = outlier_q3 + 1.5 * outlier_iqr
flagged = df[(df[outlier_column] < lower) | (df[outlier_column] > upper)]
print('Lower fence:', round(lower, 1), 'Upper fence:', round(upper, 1))
print('Flagged rows:', len(flagged))
flagged[['event_id', 'event_type', 'previous_similar_attendance', 'capacity']].head(10)


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=7caf098a-dbfc-4bf2-bb19-b4b7015348c7"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Compare a box plot with the flagged table above.
plt.figure(figsize=(8, 2.8))
sns.boxplot(x=df['previous_similar_attendance'], color='#E8A33D')
plt.xlabel('Previous similar-event attendance')
plt.title('Potential outliers deserve investigation')
plt.show()


**Evidence note 2:** Choose one flagged event. State whether it looks impossible, unusual but plausible, or unclear. Name the evidence you would need before changing or removing it.


## Level 3: Compare groups without hiding group size
A group rate can be useful, but a dramatic result from three events is not as stable as a result from eighty events. Always show the count beside the statistic.


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=9ac17276-4b97-4a64-b13c-b4b701534c6a"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Compare turnout by event type and keep the group size visible.
group_report = (df.groupby('event_type')
                  .agg(events=('event_id', 'count'),
                       median_attendance_rate=('attendance_rate', 'median'),
                       high_turnout_rate=('high_turnout_flag', 'mean'))
                  .sort_values('high_turnout_rate', ascending=False))
group_report.round(3)


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=c9d96d09-ab42-4470-b433-b4b701535ac4"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Plot the high-turnout rate. Read the table too because the chart does not show every detail.
plot_data = group_report.reset_index()
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_data, x='high_turnout_rate', y='event_type', color='#3A6EA5')
plt.xlim(0, 1)
plt.xlabel('Proportion of events with high turnout')
plt.ylabel('Event type')
plt.title('High turnout by event type')
plt.savefig('m7_turnout_by_event_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved m7_turnout_by_event_type.png')


**Evidence note 3:** Write one comparison using the words *associated with* or *in this dataset*. Include both a rate and the number of events. Do not use *caused*.


## Level 4: Relationships are clues, not proof
Correlation describes the direction and strength of a linear relationship between numeric variables. It does not establish cause, and a value near zero does not rule out every kind of relationship.


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=18defcff-8db6-41ef-b2f6-b4b7015379fc"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Use only variables known before the event for this relationship check.
pre_event_numeric = ['capacity', 'ticket_price_usd', 'promotion_days',
                     'social_posts', 'poster_count',
                     'previous_similar_attendance', 'competing_events']
corr = df[pre_event_numeric + ['high_turnout_flag']].corr(numeric_only=True)['high_turnout_flag'].drop('high_turnout_flag')
corr.sort_values(key=abs, ascending=False).round(3)


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=2c3df187-31f3-43d7-8904-b4b701538f6d"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Inspect one relationship visually. Each dot is one event.
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='previous_similar_attendance', y='attendance_rate',
                hue='high_turnout', palette={'no':'#6B7280', 'yes':'#A32638'}, alpha=.75)
plt.xlabel('Previous similar-event attendance')
plt.ylabel('Attendance rate')
plt.title('Previous attendance and current attendance rate')
plt.savefig('m7_previous_attendance_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved m7_previous_attendance_scatter.png')


**Gemini checkpoint:** Ask: `Describe the visible relationship in this scatter plot without claiming causation. Give one possible lurking variable and one check I should perform.` Then verify every statement against the axes, the dots, and the data dictionary. Record one part you accepted and one part you corrected or qualified.


## Level 5: Build the M8 feature shortlist
A candidate feature must pass three gates: it is available before the event, it is not an identifier or target-derived answer, and there is a reasonable project justification for considering it. Statistical association alone does not guarantee usefulness or fairness.


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=67486835-818d-4828-a565-b4b7015393ed"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Columns that must not be handed to the model as features.
blocked = {
    'event_id': 'identifier, not a generalizable clue',
    'event_date': 'raw date needs a justified transformation',
    'actual_weather': 'not known at advance prediction time',
    'actual_attendance': 'post-event outcome and direct leakage',
    'attendance_rate': 'directly determines the target',
    'high_turnout': 'this is the target answer',
    'high_turnout_flag': 'analysis-only numeric copy of the target answer'
}
pd.DataFrame(blocked.items(), columns=['column', 'reason'])


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=25efee95-6c22-4480-96ab-b4b70153a300"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Start a candidate table. Edit keep_for_m8 and justification after reviewing your evidence.
candidates = ['capacity', 'event_type', 'start_hour', 'day_of_week',
              'weather_forecast', 'promotion_days', 'social_posts',
              'poster_count', 'food_available', 'ticket_price_usd',
              'previous_similar_attendance', 'competing_events']
feature_table = pd.DataFrame({
    'candidate_feature': candidates,
    'available_before_event': True,
    'keep_for_m8': '',
    'evidence_or_reason': '',
    'fairness_or_proxy_question': ''
})
feature_table


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=602b16b6-54e6-4809-b318-b4b70153bd26"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Use this helper once for EACH row. The first line is an example, not a required answer.
def record_decision(feature, keep, evidence, fairness_question):
    mask = feature_table['candidate_feature'].eq(feature)
    if mask.sum() != 1:
        raise ValueError(f'Check the feature name: {feature}')
    feature_table.loc[mask, ['keep_for_m8', 'evidence_or_reason', 'fairness_or_proxy_question']] = [
        keep, evidence, fairness_question
    ]

record_decision('capacity', 'review',
                'Group summaries suggest capacity deserves testing; this does not prove usefulness.',
                'Could venue size reflect unequal access to campus resources?')

# Copy and edit record_decision(...) for every remaining candidate.
feature_table


## Final deliverable: Statistical Evidence Report
Save: (1) two labeled charts, (2) three evidence statements, (3) one limitation, (4) the completed candidate-feature table, and (5) your Gemini verification note. End with: `In Module 8, the model may test these clues, but this report does not prove that any clue causes turnout.`


<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=4c59385f-473a-4431-8fb5-b4b70153cf37"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
# Export only after every candidate has a decision and explanation.
required = ['keep_for_m8', 'evidence_or_reason', 'fairness_or_proxy_question']
incomplete = feature_table[required].apply(lambda col: col.astype(str).str.strip().eq('')).any(axis=1)
if incomplete.any():
    print('Not exported yet. Complete these rows:')
    display(feature_table.loc[incomplete, ['candidate_feature']])
else:
    feature_table.to_csv('m7_feature_candidates_for_m8.csv', index=False)
    print('Saved m7_feature_candidates_for_m8.csv')
